# Workflow Process v7: SBERT + Cosine System Integration

## Key Improvements over v6:
- **SBERT Architecture**: Mean pooling over tokens (better sentence embeddings)
- **Multi-label Support**: Assign multiple topics per chunk using cosine scores
- **Unassigned Gate**: Intelligent handling of "none of the above" cases
- **Per-class Thresholds**: Fine-grained calibration per topic (vs global temperature)
- **Class Weighting**: Better handling of imbalanced topics

## Workflow:
1. Cell 6: Load cosine-labeled score files
2. Cell 6.1-6.3: Prepare labeled/pseudo/unlabeled data with diagnostics
3. Cell 7.1: Setup SBERT training environment
4. Cell 7.2: Prepare data with multi-label cosine targets
5. Cell 7.3: Train SBERT model with unassigned gate + per-class calibration

In [ ]:
# ============================================================
# CELL 6: LOAD LABELED SCORES
# ============================================================

print(f"\n{'='*60}")
print("CHECKPOINT 6 START - LOADING LABELED SCORE FILES")
print(f"{'='*60}")

# Initialize/verify fs object
if 'fs' not in globals() or not hasattr(fs, 'folders') or not fs.folders:
    # Need to initialize or reload fs
    fs = WorkflowFileSystem(CONFIG)
    
    # Find and load the appropriate workflow
    workflow_base = Path(CONFIG["paths"]["workflow_base"])
    model_type = CONFIG['workflow']['model_type']
    topic = CONFIG['workflow']['topic']
    
    if CONFIG['workflow']['version']:
        # Find specific version (pattern: model_type-topic_DATE_version)
        pattern = f"{model_type}-{topic}_*_{CONFIG['workflow']['version']}"
        matching_dirs = list(workflow_base.glob(pattern))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[0]
    else:
        # Find most recent workflow
        pattern = f"{model_type}-{topic}_*"
        matching_dirs = sorted(list(workflow_base.glob(pattern)))
        if not matching_dirs:
            raise ValueError(f"No workflow found matching pattern: {pattern}")
        workflow_dir = matching_dirs[-1]
    
    fs.load_existing_workflow(workflow_dir)
    print(f"📂 Loaded workflow: {fs.root.name}")

# Load the three CSV files using fs.folders
high_path = fs.folders['Cosine_labeling'] / 'scores_high_confidence.csv'
low_path = fs.folders['Cosine_labeling'] / 'scores_low_confidence.csv'
no_path = fs.folders['Cosine_labeling'] / 'scores_no_confidence.csv'

if not high_path.exists() or not low_path.exists() or not no_path.exists():
    print(f"❌ Error: One or more score files not found")
    print(f"   Looking in: {fs.folders['Cosine_labeling']}")
    print(f"   - {high_path.name} {'✓' if high_path.exists() else '✗'}")
    print(f"   - {low_path.name} {'✓' if low_path.exists() else '✗'}")
    print(f"   - {no_path.name} {'✓' if no_path.exists() else '✗'}")
    raise FileNotFoundError("Required score files not found. Please run CHECKPOINT 5 first.")
else:
    high_df = pd.read_csv(high_path)
    low_df = pd.read_csv(low_path)
    no_df = pd.read_csv(no_path)
    
    print(f"✓ Loaded score files from: {fs.folders['Cosine_labeling']}")
    print(f"  High confidence: {len(high_df)} chunks")
    print(f"  Low confidence:  {len(low_df)} chunks")
    print(f"  No confidence:   {len(no_df)} chunks")
    print(f"  Total:           {len(high_df) + len(low_df) + len(no_df)} chunks")

In [ ]:
# ============================================================
# CELL 6.1: PREPARE LABELED DATA (with cosine scores for multi-label)
# ============================================================

print(f"\n{'='*60}")
print("PREPARING LABELED DATA")
print(f"{'='*60}")

# High confidence = labeled data
df_labeled = high_df.copy()
df_labeled['text'] = df_labeled['raw_text']
df_labeled['label'] = df_labeled['primary_topic']

# Create label mapping
label2id = {label: idx for idx, label in enumerate(sorted(df_labeled['label'].unique()))}
id2label = {idx: label for label, idx in label2id.items()}
df_labeled['label_id'] = df_labeled['label'].map(label2id)
df_labeled['is_pseudo'] = False

print(f"\nLabel mapping:")
for label, idx in label2id.items():
    count = (df_labeled['label'] == label).sum()
    print(f"  {idx}: {label} ({count} examples)")

print(f"\nTotal labeled examples: {len(df_labeled)}")

# IMPORTANT: Check if we have cosine score columns for multi-label targets
# Expected columns: 'scores_dict' or individual 'cos_<topic>' columns
cosine_cols = [c for c in df_labeled.columns if c.startswith('cos_')]
if cosine_cols:
    print(f"\n✓ Found {len(cosine_cols)} cosine score columns for multi-label targets:")
    print(f"  {cosine_cols[:5]}..." if len(cosine_cols) > 5 else f"  {cosine_cols}")
else:
    print(f"\n⚠ No cosine score columns found. Will use single-label mode.")
    print(f"  To enable multi-label: ensure 'cos_<topic>' columns exist in your data")

In [ ]:
# ============================================================
# CELL 6.2: PREPARE PSEUDO-LABELED & UNLABELED DATA
# ============================================================

# Use existing CONFIG or create defaults
if 'CONFIG' not in globals():
    CONFIG = {}
if 'sampling' not in CONFIG:
    CONFIG['sampling'] = {
        "unlabeled_multiplier": 3,  # Max unlabeled = labeled_size * 3
        "pseudo_multiplier": 10      # Max pseudo = labeled_size * 10
    }

print(f"\n{'='*60}")
print("PREPARING PSEUDO-LABELED & UNLABELED DATA")
print(f"{'='*60}")

# =====================
# PREPARE PSEUDO-LABELED DATA
# =====================

# Pseudo-labeled pool (low confidence predictions)
df_pseudo = low_df.copy()
df_pseudo['text'] = df_pseudo['raw_text']
df_pseudo['label'] = df_pseudo['primary_topic']
df_pseudo['label_id'] = df_pseudo['label'].map(label2id)
df_pseudo['is_pseudo'] = True

print(f"\nPseudo-labeled pool: {len(df_pseudo)} chunks")

# Sample pseudo-labeled data for balance
max_pseudo = len(df_labeled) * CONFIG["sampling"]["pseudo_multiplier"]
pseudo_cols = ['text', 'label', 'label_id', 'is_pseudo'] + [c for c in df_pseudo.columns if c.startswith('cos_')]
pseudo_cols = [c for c in pseudo_cols if c in df_pseudo.columns]

if len(df_pseudo) > max_pseudo:
    df_pseudo_sampled = df_pseudo[pseudo_cols].sample(n=max_pseudo, random_state=42)
    print(f"  Sampled: {len(df_pseudo_sampled)} (to maintain balance)")
else:
    df_pseudo_sampled = df_pseudo[pseudo_cols].copy()
    print(f"  Using all: {len(df_pseudo_sampled)}")

# =====================
# PREPARE UNLABELED DATA
# =====================

# Unlabeled pool (no confidence predictions)
df_unlabeled = no_df[['raw_text']].copy()
df_unlabeled.rename(columns={'raw_text': 'text'}, inplace=True)
df_unlabeled['label'] = 'UNLABELED'
df_unlabeled['label_id'] = -1
df_unlabeled['is_pseudo'] = False

# Clean: remove empty/null text
df_unlabeled = df_unlabeled[df_unlabeled['text'].notna()].copy()
df_unlabeled = df_unlabeled[df_unlabeled['text'].astype(str).str.strip() != ''].copy()

print(f"\nUnlabeled pool: {len(df_unlabeled)} chunks")

# Sample unlabeled data for balance
max_unlabeled = len(df_labeled) * CONFIG["sampling"]["unlabeled_multiplier"]
if len(df_unlabeled) > max_unlabeled:
    df_unlabeled_sampled = df_unlabeled.sample(n=max_unlabeled, random_state=42)
    print(f"  Sampled: {len(df_unlabeled_sampled)} (to maintain balance)")
else:
    df_unlabeled_sampled = df_unlabeled.copy()
    print(f"  Using all: {len(df_unlabeled_sampled)}")

# =====================
# SUMMARY
# =====================

print(f"\n{'='*60}")
print("DATA PREPARATION SUMMARY")
print(f"{'='*60}")
print(f"  Labeled:     {len(df_labeled)}")
print(f"  Pseudo:      {len(df_pseudo_sampled)}")
print(f"  Unlabeled:   {len(df_unlabeled_sampled)}")
print(f"  Total pool:  {len(df_labeled) + len(df_pseudo_sampled) + len(df_unlabeled_sampled)}")

In [ ]:
# ============================================================
# CELL 6.3: CREATE DATASET OPTIONS & TRAIN/VAL SPLIT
# (Same as v6 - keeping existing workflow)
# ============================================================

from sklearn.model_selection import train_test_split

print(f"\n{'='*60}")
print("CREATING DATASET OPTIONS & TRAIN/VAL SPLIT")
print(f"{'='*60}")

# =====================
# STEP 1: GROUP DATA INTO OPTIONS FIRST
# =====================

print(f"\nStep 1: Grouping data into options...")

# =====================
# OPTION 1: LABELED ONLY
# =====================
data_opt1 = df_labeled.copy()

# =====================
# OPTION 2: LABELED + PSEUDO-LABELED
# =====================
data_opt2 = pd.concat([df_labeled, df_pseudo_sampled], ignore_index=True)

# =====================
# OPTION 3: LABELED + UNLABELED
# =====================
data_opt3 = pd.concat([df_labeled, df_unlabeled_sampled], ignore_index=True)

# =====================
# OPTION 4: ALL (LABELED + PSEUDO + UNLABELED)
# =====================
data_opt4 = pd.concat([df_labeled, df_pseudo_sampled, df_unlabeled_sampled], ignore_index=True)

print(f"  Option 1 (Labeled only):          {len(data_opt1):>6} examples")
print(f"  Option 2 (Labeled + Pseudo):      {len(data_opt2):>6} examples")
print(f"  Option 3 (Labeled + Unlabeled):   {len(data_opt3):>6} examples")
print(f"  Option 4 (All) ⭐ RECOMMENDED:    {len(data_opt4):>6} examples")

# =====================
# STEP 2: SPLIT EACH OPTION INTO TRAIN/VAL
# =====================

print(f"\nStep 2: Splitting each option into train/val...")

def split_with_stratification(data, option_name):
    """Split data into train/val, using stratification if possible."""
    labeled_data = data[data['label'] != 'UNLABELED'].copy()
    unlabeled_data = data[data['label'] == 'UNLABELED'].copy()
    
    if len(labeled_data) > 0:
        topic_counts = labeled_data['label'].value_counts()
        can_stratify = all(topic_counts >= 2)
        
        if can_stratify:
            train_labeled, val_labeled = train_test_split(
                labeled_data, test_size=0.2, stratify=labeled_data['label'], random_state=42
            )
            print(f"  {option_name}: ✓ Stratified split")
        else:
            train_labeled, val_labeled = train_test_split(
                labeled_data, test_size=0.2, random_state=42
            )
            print(f"  {option_name}: ⚠ Random split (some topics < 2 examples)")
        
        if len(unlabeled_data) > 0:
            train_data = pd.concat([train_labeled, unlabeled_data], ignore_index=True)
            print(f"      Added {len(unlabeled_data)} unlabeled to training")
        else:
            train_data = train_labeled
        
        val_data = val_labeled
    else:
        train_data = data
        val_data = data.head(0)
        print(f"  {option_name}: ⚠ No labeled data for validation")
    
    return train_data, val_data

# Split each option
train_opt1, val_opt1 = split_with_stratification(data_opt1, "Option 1")
train_opt2, val_opt2 = split_with_stratification(data_opt2, "Option 2")
train_opt3, val_opt3 = split_with_stratification(data_opt3, "Option 3")
train_opt4, val_opt4 = split_with_stratification(data_opt4, "Option 4")

print(f"\n✓ All dataset options prepared")

In [ ]:
# ============================================================
# CELL 7.1: SETUP SBERT TRAINING ENVIRONMENT
# ============================================================

print(f"\n{'='*60}")
print("SETTING UP SBERT TRAINING ENVIRONMENT (v7)")
print(f"{'='*60}")

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader
    from transformers import (
        AutoModel,
        AutoTokenizer,
        AutoConfig,
        TrainingArguments,
        Trainer,
        DataCollatorWithPadding,
        EvalPrediction
    )
    from transformers.modeling_outputs import SequenceClassifierOutput
    from datasets import Dataset
    import numpy as np
    from sklearn.metrics import (
        accuracy_score,
        precision_recall_fscore_support,
        f1_score,
        classification_report
    )
    
    print("✓ All required libraries available")
    
    # Check GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    TRAINING_AVAILABLE = True
    
except ImportError as e:
    print(f"⚠ Missing library: {e}")
    print("  Install: pip install transformers datasets torch sklearn")
    TRAINING_AVAILABLE = False

# ============================================================
# SBERT ARCHITECTURE COMPONENTS
# ============================================================

if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("DEFINING SBERT ARCHITECTURE")
    print(f"{'='*60}")
    
    class MeanPooling(nn.Module):
        """Mean pooling over all tokens (weighted by attention mask)."""
        def forward(self, last_hidden_state, attention_mask):
            mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)  # [B,S,1]
            summed = (last_hidden_state * mask).sum(dim=1)  # [B,H]
            counts = mask.sum(dim=1).clamp(min=1e-9)  # [B,1]
            return summed / counts
    
    class SBERTClassifier(nn.Module):
        """SBERT-style classifier with mean pooling + configurable head."""
        def __init__(self, base_name: str, num_labels: int, use_multi_label: bool = False, dropout: float = 0.1):
            super().__init__()
            self.encoder = AutoModel.from_pretrained(base_name)
            hidden = self.encoder.config.hidden_size
            self.pool = MeanPooling()
            self.dropout = nn.Dropout(dropout)
            self.classifier = nn.Linear(hidden, num_labels)
            
            # Store config
            self.config = AutoConfig.from_pretrained(base_name)
            self.config.num_labels = num_labels
            self.config.problem_type = "multi_label_classification" if use_multi_label else "single_label_classification"
            self.use_multi_label = use_multi_label
            
            self.id2label = None
            self.label2id = None
        
        def forward(self, input_ids=None, attention_mask=None, **kwargs):
            out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
            sent = self.pool(out.last_hidden_state, attention_mask)
            sent = self.dropout(sent)
            logits = self.classifier(sent)
            return SequenceClassifierOutput(logits=logits)
    
    print("✓ SBERT architecture defined:")
    print("  - MeanPooling: Average over all tokens (vs [CLS] token)")
    print("  - SBERTClassifier: BERT encoder + mean pooling + classification head")
    print("  - Supports both single-label and multi-label classification")

In [ ]:
if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("PREPARING DATA FOR SBERT TRAINING")
    print(f"{'='*60}")

    # =====================
    # 1) PICK DATASET OPTION
    # =====================

    if 'training' not in CONFIG:
        CONFIG['training'] = {'dataset_option': 'option4'}

    dataset_option = CONFIG['training'].get('dataset_option', 'option4')

    if dataset_option == 'option1':
        train_dataset, val_dataset = train_opt1, val_opt1
    elif dataset_option == 'option2':
        train_dataset, val_dataset = train_opt2, val_opt2
    elif dataset_option == 'option3':
        train_dataset, val_dataset = train_opt3, val_opt3
    else:
        train_dataset, val_dataset = train_opt4, val_opt4

    print(f"\nUsing {dataset_option}:")
    print(f"  Train: {len(train_dataset)} examples")
    print(f"  Val:   {len(val_dataset)} examples")

    # =====================
    # 2) DETECT MULTI-LABEL MODE (check for cosine columns)
    # =====================

    train_cols = list(train_dataset.columns)
    cos_cols = [c for c in train_cols if isinstance(c, str) and c.startswith("cos_")]

    # Keep only columns that exist in both train and val
    cos_cols = [c for c in cos_cols if c in val_dataset.columns]

    USE_MULTI_LABEL = len(cos_cols) == len(label2id)

    if USE_MULTI_LABEL:
        print(f"\n✓ MULTI-LABEL MODE ENABLED")
        print(f"  Found {len(cos_cols)} cosine score columns matching {len(label2id)} topics")
        print(f"  Will train multi-label classifier using cosine scores as soft targets")
    else:
        print(f"\n✓ SINGLE-LABEL MODE")
        print(f"  Found {len(cos_cols)} cosine columns (expected {len(label2id)})")
        print(f"  Will train single-label classifier using primary topic labels")

    # =====================
    # 3) LOAD MODEL + TOKENIZER
    # =====================

    from pathlib import Path

    SBERT_MODEL_NAME = "GroNLP/bert-base-dutch-cased"  # Dutch-capable BERT

    if CONFIG.get('model', {}).get('use_pretrained') and CONFIG.get('paths', {}).get('pretrained_model_path'):
        model_path = CONFIG['paths']['pretrained_model_path']
        if Path(model_path).exists():
            model_name = model_path
            print(f"\n✓ Loading pretrained model from: {model_path}")
        else:
            model_name = SBERT_MODEL_NAME
            print(f"\n⚠ Pretrained path not found, using base: {SBERT_MODEL_NAME}")
    else:
        model_name = SBERT_MODEL_NAME
        print(f"\n✓ Using base model: {SBERT_MODEL_NAME}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = SBERTClassifier(
        base_name=model_name,
        num_labels=len(label2id),
        use_multi_label=USE_MULTI_LABEL,
        dropout=0.1
    )
    model.id2label = id2label
    model.label2id = label2id
    model.to(device)

    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"  Architecture: SBERT with mean pooling")

    # =====================
    # 4) COMPUTE CLASS WEIGHTS (for imbalanced topics)
    # =====================

    train_labels = train_dataset[train_dataset['label_id'] != -1]['label_id'].values
    label_counts = np.bincount(train_labels, minlength=len(label2id))

    # Inverse frequency weighting with sqrt smoothing
    total = label_counts.sum()
    class_weights = np.sqrt(total / (label_counts + 1))  # +1 to avoid division by zero
    class_weights = class_weights / class_weights.mean()  # normalize
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

    print(f"\n✓ Computed class weights (sqrt inverse frequency):")
    for idx, (label, weight) in enumerate(zip(id2label.values(), class_weights)):
        print(f"  {label}: {weight:.3f} (n={label_counts[idx]})")

    # =====================
    # 5) PREPARE DATASETS - FIXED VERSION
    # =====================

    def tokenize_function(examples):
        return tokenizer(examples['text'], truncation=True, max_length=512)

    # Filter out unlabeled data from training
    train_labeled = train_dataset[train_dataset['label_id'] != -1].copy().reset_index(drop=True)
    val_labeled = val_dataset[val_dataset['label_id'] != -1].copy().reset_index(drop=True)

    # Select columns based on mode
    if USE_MULTI_LABEL:
        print("\n[Multi-label] Creating binary targets from cosine scores...")

        # Create multi-label targets from cosine scores (threshold at 0.5)
        # IMPORTANT: Convert to numpy array BEFORE creating HF dataset
        threshold = 0.5

        train_label_array = np.zeros((len(train_labeled), len(label2id)), dtype=np.float32)
        for i, (idx, row) in enumerate(train_labeled.iterrows()):
            for j, col in enumerate(cos_cols):
                score = row[col] if col in row and pd.notna(row[col]) else 0.0
                train_label_array[i, j] = 1.0 if score >= threshold else 0.0

        val_label_array = np.zeros((len(val_labeled), len(label2id)), dtype=np.float32)
        for i, (idx, row) in enumerate(val_labeled.iterrows()):
            for j, col in enumerate(cos_cols):
                score = row[col] if col in row and pd.notna(row[col]) else 0.0
                val_label_array[i, j] = 1.0 if score >= threshold else 0.0

        # Create datasets with text and labels
        train_data_dict = {
            'text': train_labeled['text'].tolist(),
            'labels': train_label_array  # numpy array, not list of lists
        }
        val_data_dict = {
            'text': val_labeled['text'].tolist(),
            'labels': val_label_array
        }

        hf_train = Dataset.from_dict(train_data_dict)
        hf_val = Dataset.from_dict(val_data_dict)

        print(f"  Train labels shape: {train_label_array.shape}")
        print(f"  Val labels shape: {val_label_array.shape}")
    else:
        # Single-label: just use label_id
        hf_train = Dataset.from_pandas(train_labeled[['text', 'label_id']].reset_index(drop=True))
        hf_val = Dataset.from_pandas(val_labeled[['text', 'label_id']].reset_index(drop=True))
        hf_train = hf_train.rename_column('label_id', 'labels')
        hf_val = hf_val.rename_column('label_id', 'labels')

    # Tokenize
    hf_train = hf_train.map(tokenize_function, batched=True, remove_columns=['text'])
    hf_val = hf_val.map(tokenize_function, batched=True, remove_columns=['text'])

    # Set format to torch
    hf_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    hf_val.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

    # Data collator
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    print(f"\n✓ Data prepared:")
    print(f"  Train: {len(hf_train)} examples")
    print(f"  Val:   {len(hf_val)} examples")
    print(f"  Mode:  {'Multi-label' if USE_MULTI_LABEL else 'Single-label'}")

else:
    print("⚠ Skipping data preparation - transformers library not available")




In [ ]:
# ============================================================
# CELL 7.3: TRAIN SBERT MODEL WITH UNASSIGNED GATE + CALIBRATION
# ============================================================

 TRAIN SBERT MODEL WITH UNASSIGNED GATE + CALIBRATION
# ============================================================

if TRAINING_AVAILABLE:
    print(f"\n{'='*60}")
    print("TRAINING SBERT MODEL (v7)")
    print(f"{'='*60}")

    # =====================
    # 1) UNASSIGNED GATE CONFIGURATION
    # =====================

    USE_UNASSIGNED_GATE = True
    UNASSIGNED_LABEL_CANDIDATES = ["Unassigned", "Uncategorised", "Uncategorized", "Other"]

    # Find unassigned label index
    UNASSIGNED_IDX = None
    for candidate in UNASSIGNED_LABEL_CANDIDATES:
        if candidate in label2id:
            UNASSIGNED_IDX = label2id[candidate]
            break

    if UNASSIGNED_IDX is not None:
        print(f"\n✓ Unassigned gate ENABLED")
        print(f"  Unassigned label: '{list(label2id.keys())[UNASSIGNED_IDX]}' (index {UNASSIGNED_IDX})")
        print(f"  Mode: Hard threshold (will be calibrated post-training)")
    else:
        USE_UNASSIGNED_GATE = False
        print(f"\n⚠ Unassigned gate DISABLED (no matching label found)")
        print(f"  Looked for: {UNASSIGNED_LABEL_CANDIDATES}")

    # =====================
    # 2) METRICS FUNCTION
    # =====================

    def compute_metrics(eval_pred):
        if isinstance(eval_pred, EvalPrediction):
            logits, labels = eval_pred.predictions, eval_pred.label_ids
        else:
            logits, labels = eval_pred

        if USE_MULTI_LABEL:
            # Multi-label metrics
            probs = torch.sigmoid(torch.tensor(logits)).numpy()
            preds = (probs >= 0.5).astype(int)
            labels_int = labels.astype(int)

            return {
                'accuracy': accuracy_score(labels_int, preds),
                'precision': precision_recall_fscore_support(labels_int, preds, average='weighted', zero_division=0)[0],
                'recall': precision_recall_fscore_support(labels_int, preds, average='weighted', zero_division=0)[1],
                'f1': f1_score(labels_int, preds, average='weighted', zero_division=0)
            }
        else:
            # Single-label metrics
            preds = np.argmax(logits, axis=1)
            precision, recall, f1, _ = precision_recall_fscore_support(
                labels, preds, average='weighted', zero_division=0
            )
            return {
                'accuracy': accuracy_score(labels, preds),
                'precision': precision,
                'recall': recall,
                'f1': f1
            }

    # =====================
    # 3) CUSTOM TRAINER WITH WEIGHTED LOSS - FIXED VERSION
    # =====================

    class WeightedLossTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            # Extract labels safely
            if "labels" in inputs:
                labels = inputs.pop("labels")
            elif "label" in inputs:
                labels = inputs.pop("label")
            else:
                raise KeyError("Neither 'labels' nor 'label' found in inputs")

            # Forward pass
            outputs = model(**inputs)
            logits = outputs.logits

            if USE_MULTI_LABEL:
                # Multi-label: BCEWithLogitsLoss with class weights
                labels = labels.float()
                loss_fn = nn.BCEWithLogitsLoss(pos_weight=class_weights)
                loss = loss_fn(logits, labels)
            else:
                # Single-label: CrossEntropyLoss with class weights
                labels = labels.long()
                loss_fn = nn.CrossEntropyLoss(weight=class_weights)
                loss = loss_fn(logits, labels)

            return (loss, outputs) if return_outputs else loss

    # =====================
    # 4) TRAINING ARGUMENTS
    # =====================

    model_output_dir = str(fs.folders['Model_finetuning']) if 'fs' in globals() else "./sbert_model"

    # Use CONFIG training params or defaults
    training_config = CONFIG.get('training', {})

    training_args = TrainingArguments(
        output_dir=model_output_dir,
        num_train_epochs=training_config.get('num_epochs', 5),
        per_device_train_batch_size=training_config.get('batch_size_train', 16),
        per_device_eval_batch_size=training_config.get('batch_size_eval', 32),
        learning_rate=training_config.get('learning_rate', 2e-5),
        weight_decay=training_config.get('weight_decay', 0.01),
        warmup_ratio=training_config.get('warmup_ratio', 0.1),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=f"{model_output_dir}/logs",
        logging_strategy="steps",
        logging_steps=50,
        report_to=None,
        fp16=torch.cuda.is_available(),
        seed=42,
        push_to_hub=False,
    )

    print(f"\nTraining configuration:")
    print(f"  Epochs: {training_args.num_train_epochs}")
    print(f"  Batch size (train): {training_args.per_device_train_batch_size}")
    print(f"  Batch size (eval): {training_args.per_device_eval_batch_size}")
    print(f"  Learning rate: {training_args.learning_rate}")
    print(f"  FP16: {training_args.fp16}")

    # =====================
    # 5) CREATE TRAINER AND TRAIN
    # =====================

    trainer = WeightedLossTrainer(
        model=model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print(f"\n{'='*60}")
    print(f"Starting training...")
    print(f"{'='*60}\n")

    train_result = trainer.train()

    print(f"\n{'='*60}")
    print("TRAINING COMPLETE")
    print(f"{'='*60}")

    # =====================
    # 6) EVALUATE
    # =====================

    eval_results = trainer.evaluate()

    print(f"\nValidation Results:")
    print(f"  Accuracy:  {eval_results['eval_accuracy']:.4f}")
    print(f"  Precision: {eval_results['eval_precision']:.4f}")
    print(f"  Recall:    {eval_results['eval_recall']:.4f}")
    print(f"  F1 Score:  {eval_results['eval_f1']:.4f}")

    # =====================
    # 7) PER-CLASS THRESHOLD CALIBRATION (SBERT-style)
    # =====================

    print(f"\n{'='*60}")
    print("CALIBRATING PER-CLASS THRESHOLDS")
    print(f"{'='*60}")

    val_pred = trainer.predict(hf_val)
    val_logits = val_pred.predictions
    val_labels = val_pred.label_ids

    if USE_MULTI_LABEL:
        # Multi-label: find best threshold per class
        probs = torch.sigmoid(torch.tensor(val_logits)).numpy()

        def find_best_thresholds(y_true, probs, grid=np.linspace(0.05, 0.95, 19)):
            C = y_true.shape[1]
            best = [0.5] * C
            for c in range(C):
                y = y_true[:, c]
                if y.sum() == 0:
                    continue
                p = probs[:, c]
                best_f1, best_t = 0.0, 0.5
                for t in grid:
                    f1 = f1_score(y, (p >= t).astype(int), zero_division=0)
                    if f1 > best_f1:
                        best_f1, best_t = f1, t
                best[c] = float(best_t)
            return best

        thresholds = find_best_thresholds(val_labels.astype(int), probs)

        print(f"\nPer-class thresholds (optimized for F1):")
        for idx, (label, thr) in enumerate(zip(id2label.values(), thresholds)):
            print(f"  {label}: {thr:.3f}")

        # Save thresholds
        threshold_info = {
            "topic_names": list(id2label.values()),
            "thresholds": thresholds,
            "use_unassigned_gate": USE_UNASSIGNED_GATE,
            "unassigned_idx": UNASSIGNED_IDX,
            "mode": "multi_label"
        }

        if 'fs' in globals():
            fs.save_data(threshold_info, "sbert_thresholds", "Model_finetuning", "json")
            print("\n✓ Saved: Model_finetuning/sbert_thresholds.json")

    else:
        # Single-label: temperature scaling
        def nll_with_T(T):
            T = max(0.05, float(T))
            z = torch.tensor(val_logits) / T
            logp = torch.log_softmax(z, dim=1).numpy()
            return -float(np.mean([logp[i, val_labels[i]] for i in range(len(val_labels))]))

        T = 1.0
        for _ in range(20):
            cands = [max(0.05, T * f) for f in (0.5, 0.75, 1.0, 1.25, 1.5)]
            losses = [nll_with_T(c) for c in cands]
            T = cands[int(np.argmin(losses))]

        p_cal = torch.softmax(torch.tensor(val_logits) / T, dim=1).numpy()
        pmax_cal = p_cal.max(axis=1)

        # Tier thresholds (30% high, 30% medium, 40% low/irrelevant)
        hi_tau = float(np.quantile(pmax_cal, 0.70))
        med_tau = float(np.quantile(pmax_cal, 0.40))

        print(f"\nTemperature scaling:")
        print(f"  Optimal T: {T:.3f}")
        print(f"  High confidence threshold: {hi_tau:.3f}")
        print(f"  Medium confidence threshold: {med_tau:.3f}")

        calib_info = {
            "temperature_T": T,
            "tier_thresholds": {
                "HIGH": hi_tau,
                "MEDIUM": med_tau,
                "LOW_or_IRRELEVANT": 0.0
            },
            "mode": "single_label"
        }

        if 'fs' in globals():
            fs.save_data(calib_info, "sbert_temperature_and_tiers", "Model_finetuning", "json")
            print("\n✓ Saved: Model_finetuning/sbert_temperature_and_tiers.json")

    # =====================
    # 8) SAVE MODEL + METRICS
    # =====================

    trainer.save_model(model_output_dir)
    tokenizer.save_pretrained(model_output_dir)

    metrics = {
        "train_loss": float(train_result.training_loss),
        "train_runtime": train_result.metrics['train_runtime'],
        "eval_accuracy": eval_results['eval_accuracy'],
        "eval_precision": eval_results['eval_precision'],
        "eval_recall": eval_results['eval_recall'],
        "eval_f1": eval_results['eval_f1'],
        "eval_loss": eval_results['eval_loss'],
        "num_train_examples": len(hf_train),
        "num_eval_examples": len(hf_val),
        "num_epochs": training_args.num_train_epochs,
        "dataset_used": dataset_option,
        "architecture": "SBERT",
        "mode": "multi_label" if USE_MULTI_LABEL else "single_label",
        "unassigned_gate": USE_UNASSIGNED_GATE
    }

    if 'fs' in globals():
        fs.save_data(metrics, "sbert_training_metrics", "Model_finetuning", "json")
        fs.save_config("checkpoint7_sbert_trained")
        print("✓ Saved: Model_finetuning/sbert_training_metrics.json")
        print("✓ Checkpoint saved: checkpoint7_sbert_trained")

    print(f"\n{'='*60}")
    print("✓ SBERT TRAINING COMPLETE")
    print(f"{'='*60}")
    print(f"\nModel architecture: SBERT with mean pooling")
    print(f"Classification mode: {'Multi-label' if USE_MULTI_LABEL else 'Single-label'}")
    print(f"Unassigned gate: {'Enabled' if USE_UNASSIGNED_GATE else 'Disabled'}")
    print(f"Class weighting: Enabled (sqrt inverse frequency)")
    print(f"Per-class thresholds: {'Calibrated' if USE_MULTI_LABEL else 'Temperature scaled'}")

else:
    print("⚠ Skipping training - required libraries not available")
